# XGBoost Nutrient-Aware Pathogen Suppression Predictor 0101

Predicts community_suppression using ONLY:
  - Species membership (binary, 100 cols)
  - Nutrient environment (R0 concentrations + active flags + diversity metadata)

Input file: community_index_v8_backup.xls  (168 cols, 30,000 rows)
Target:     community_suppression          (float in [0,1])

# Changes in the second run (applied to all other codes too):

1. More error metrics for per-condition breakdown as R-squared does not give an accurate picture on the slices
2. GroupKFold train-test splitting analysis added
3. Multi seed stability evaluation
4. Along with Ridge, Lassio and ElasticNet baselines also added
5. More error metrics for the final model also, and training curves added

# Third and Final Run (changes applied to all other codes too):

1. Fixed final dataset
2. GroupShuffleSplit added
3. Crucial data leakage fixes applied
   


In [2]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, LassoCV, ElasticNetCV
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupShuffleSplit, GroupKFold
from scipy.stats import pearsonr, spearmanr


In [3]:

df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Codes\MiCRM_Final_Dataset\community_index_v10.csv")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nTarget (community_suppression) statistics:")
print(df['community_suppression'].describe().round(4))

print("\nRows per diversity level:")
print(df['diversity_level'].value_counts().sort_index())

print("\nRows per community size:")
print(df['community_size'].value_counts().sort_index())

Dataset shape: 75,000 rows × 168 columns

Target (community_suppression) statistics:
count    75000.0000
mean         0.5166
std          0.3328
min          0.0000
25%          0.2331
50%          0.4961
75%          0.8426
max          0.9995
Name: community_suppression, dtype: float64

Rows per diversity level:
diversity_level
1     15000
2     15000
4     15000
8     15000
16    15000
Name: count, dtype: int64

Rows per community size:
community_size
12    15000
16    15000
24    15000
32    15000
48    15000
Name: count, dtype: int64


In [4]:
# Species membership columns (100 binary features) 
# Each column is 1 if that species is present in the community, 0 otherwise.
species_cols = [f"Species_{i:04d}" for i in range(2, 102)]


# R0 nutrient concentration columns (20 features)
# R0_c1 through R0_c20: initial concentration of each carbon source.
# Most are 0 (inactive). Active ones are ~20/n_active
r0_cols = [f"R0_c{k}" for k in range(1, 21)]


# Active flags (20 binary features)
# active_c1 through active_c20: 1 if that carbon source is supplied, 0 if not.
# These are redundant with R0 (if R0>0 then active=1) but are kept anyways
# CLARIFY ONCE: DO WE NEED TO KEEP THIS?
active_cols = [f"active_c{k}" for k in range(1, 21)]

# Structural environment features (2 features)
# community_size:  how many species are in this community (12, 16, 24, 32, 48).
# diversity_level: how many carbon sources are active (1, 2, 4, 8, 16).

structural_cols = ['community_size', 'diversity_level']

# Combine into full feature matrix

feature_cols = species_cols + r0_cols + active_cols + structural_cols
print(f"Total features: {len(feature_cols)}")
print(f"  Species membership:   {len(species_cols)}")
print(f"  R0 concentrations:    {len(r0_cols)}")
print(f"  Active flags:         {len(active_cols)}")
print(f"  Structural (size/div):{len(structural_cols)}")


X = df[feature_cols]
y = df['community_suppression']

print(f"\nX shape: {X.shape}    y shape: {y.shape}")
print(f"y range: [{y.min():.4f}, {y.max():.4f}]   mean: {y.mean():.4f}")

Total features: 142
  Species membership:   100
  R0 concentrations:    20
  Active flags:         20
  Structural (size/div):2

X shape: (75000, 142)    y shape: (75000,)
y range: [0.0000, 0.9995]   mean: 0.5166


In [5]:
strat_key = df["community_size"].astype(str) + "_" + df["diversity_level"].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key)

# broth label for each TRAIN row, positionally aligned with X_train.
# df.loc[X_train.index, ...] reorders the broth column to match X_train's rows.
groups_train = df.loc[X_train.index, "broth_id"].values


def inner_holdout(X_tr, y_tr, groups_tr, frac=0.15, seed=42):
    """Carve a group-aware validation set out of training data for early stopping.

    GroupShuffleSplit makes one split that puts whole broths (never split) into
    either the fit part or the validation part. We only need one validation set
    for early stopping, so one split is exactly right .
    .split(...) yields (fit_idx, val_idx) as POSITIONAL indices into X_tr, so we
    take them with .iloc.
    """
    gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
    fit_idx, val_idx = next(gss.split(X_tr, y_tr, groups=groups_tr))
    return (X_tr.iloc[fit_idx], y_tr.iloc[fit_idx],
            X_tr.iloc[val_idx], y_tr.iloc[val_idx])


In [9]:
# baseline models (lasso and elastic net added instead of just ridge)

from sklearn.linear_model import RidgeCV

print("Baseline comparisons")

# Baseline 1: Global mean predictor 
y_mean      = y_train.mean()
y_pred_mean = np.full_like(y_test, fill_value=y_mean, dtype=float)
r2_mean     = r2_score(y_test, y_pred_mean)
print(f"Baseline (global mean predictor):  R² = {r2_mean:.4f}")


scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Baseline 2: Ridge 
ridge_alphas = np.logspace(-3, 3, 100)

ridge_cv = RidgeCV(alphas=ridge_alphas, cv=5, scoring='r2')
ridge_cv.fit(X_train_scaled, y_train)

y_pred_ridge = ridge_cv.predict(X_test_scaled)
r2_ridge = r2_score(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)

print(f"Ridge (alpha={ridge_cv.alpha_:.5f}):         R² = {r2_ridge:.4f}   MAE = {mae_ridge:.4f}")

# Baseline 3: Lasso with cross-validated alpha selection.
# LassoCV tries 50 alpha values on a log scale and picks the one with
# the lowest 5-fold CV error. max_iter=10000 prevents convergence warnings
# with 142 features.
lasso_cv       = LassoCV(cv=5, max_iter=10000, n_alphas=50, random_state=42, n_jobs=-1)
lasso_cv.fit(X_train_scaled, y_train)
y_pred_lasso   = lasso_cv.predict(X_test_scaled)
r2_lasso       = r2_score(y_test, y_pred_lasso)
mae_lasso      = mean_absolute_error(y_test, y_pred_lasso)
n_lasso_kept   = int(np.sum(lasso_cv.coef_ != 0))
print(f"Lasso (alpha={lasso_cv.alpha_:.5f}):       R² = {r2_lasso:.4f}   MAE = {mae_lasso:.4f}   features kept = {n_lasso_kept}/{len(feature_cols)}")

# Baseline 4: Elastic Net with cross-validated alpha and l1_ratio selection.
# l1_ratio controls the L1/L2 mix: 1.0 = pure Lasso, 0.0 = pure Ridge.
enet_cv        = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95],
                               cv=5, max_iter=10000, n_alphas=50,
                               random_state=42, n_jobs=-1)
enet_cv.fit(X_train_scaled, y_train)
y_pred_enet    = enet_cv.predict(X_test_scaled)
r2_enet        = r2_score(y_test, y_pred_enet)
mae_enet       = mean_absolute_error(y_test, y_pred_enet)
n_enet_kept    = int(np.sum(enet_cv.coef_ != 0))
print(f"ElasticNet (alpha={enet_cv.alpha_:.5f}, l1={enet_cv.l1_ratio_:.2f}): R² = {r2_enet:.4f}   MAE = {mae_enet:.4f}   features kept = {n_enet_kept}/{len(feature_cols)}")

# Which species did Lasso keep?
lasso_coef      = pd.Series(lasso_cv.coef_, index=feature_cols)
lasso_nonzero   = lasso_coef[lasso_coef != 0].sort_values(key=abs, ascending=False)
sp_kept         = int((lasso_coef[species_cols] != 0).sum())
r0_kept         = int((lasso_coef[r0_cols] != 0).sum())
print(f"\nLasso feature selection: Species kept={sp_kept}/100   R0 kept={r0_kept}/20")
print("Lasso top 10 features by |coefficient|:")
for feat, coef in lasso_nonzero.head(10).items():
    ftype = ("Species" if feat in species_cols else "R0" if feat in r0_cols
             else "Active" if feat in active_cols else "Structural")
    print(f"  {feat:<22}  coef={coef:>8.5f}  [{ftype}]")

Baseline comparisons
Baseline (global mean predictor):  R² = -0.0000
Ridge (alpha=0.00100):         R² = 0.6425   MAE = 0.1537
Lasso (alpha=0.00013):       R² = 0.6426   MAE = 0.1536   features kept = 134/142
ElasticNet (alpha=0.00014, l1=0.95): R² = 0.6426   MAE = 0.1536   features kept = 134/142

Lasso feature selection: Species kept=94/100   R0 kept=19/20
Lasso top 10 features by |coefficient|:
  R0_c1                   coef=-0.14499  [R0]
  R0_c10                  coef=-0.08714  [R0]
  R0_c2                   coef=-0.08522  [R0]
  active_c4               coef= 0.08192  [Active]
  R0_c4                   coef=-0.07896  [R0]
  active_c1               coef= 0.07421  [Active]
  active_c10              coef= 0.06907  [Active]
  active_c8               coef=-0.06890  [Active]
  active_c6               coef=-0.06703  [Active]
  active_c13              coef=-0.06044  [Active]


In [6]:
# xgboost initial

Xi_fit, yi_fit, Xi_val, yi_val = inner_holdout(X_train, y_train, groups_train)

model_init = xgb.XGBRegressor(
    n_estimators=500, 
    max_depth=6, 
    learning_rate=0.05, 
    subsample=0.8,
    colsample_bytree=0.8, 
    min_child_weight=3, 
    reg_lambda=1.0,
    random_state=42, 
    n_jobs=-1, 
    eval_metric="rmse",
    early_stopping_rounds=30, 
    verbosity=0)
# eval_set is the INNER validation set (from training broths). X_test is absent.
model_init.fit(Xi_fit, yi_fit, eval_set=[(Xi_val, yi_val)], verbose=False)
y_pred_init = model_init.predict(X_test)
r2_init  = r2_score(y_test, y_pred_init)
mae_init = mean_absolute_error(y_test, y_pred_init)
print(f"\nXGBoost (initial):  R2 = {r2_init:.4f}  MAE = {mae_init:.4f}  "
      f"trees used = {model_init.best_iteration}")


XGBoost (initial):  R2 = 0.8728  MAE = 0.0700  trees used = 498


In [7]:
# hyperparameter tuning

param_grid = {                                    # tuning was already performed in a test run of this final updated code, so using those params
    "max_depth":        [7],
    "learning_rate":    [0.1],
    "subsample":        [0.6],
    "colsample_bytree": [0.9],
}

# GroupKFold(3): 3 CV folds where no broth appears in two folds. There is no
# shuffle/random_state because it partitions by GROUP, not by random rows.
group_cv = GroupKFold(n_splits=3)

# base_estimator has NO early stopping and NO eval_set, so tuning cannot peek
# at anything. n_estimators kept low for speed; the winner is retrained bigger.
base_estimator = xgb.XGBRegressor(
    n_estimators=200, min_child_weight=3, reg_lambda=1.0,
    random_state=42, n_jobs=-1, eval_metric="rmse", verbosity=0)

grid_search = GridSearchCV(
    estimator=base_estimator, param_grid=param_grid, cv=group_cv,
    scoring="neg_mean_squared_error", n_jobs=-1, verbose=1, refit=True)
# The 3rd argument groups=groups_train is what makes GridSearchCV honour the
# broth grouping. Without it, GridSearchCV silently splits by random rows.
grid_search.fit(X_train, y_train, groups=groups_train)
best_params = grid_search.best_params_
print("\nBest parameters (group-aware CV):")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

Fitting 3 folds for each of 1 candidates, totalling 3 fits

Best parameters (group-aware CV):
  colsample_bytree: 0.9
  learning_rate: 0.1
  max_depth: 7
  subsample: 0.6
Best CV RMSE: 0.2282


In [10]:
# final model 

Xf_fit, yf_fit, Xf_val, yf_val = inner_holdout(X_train, y_train, groups_train)

model_final = xgb.XGBRegressor(
    **best_params, n_estimators=1000, min_child_weight=3, reg_lambda=1.0,
    random_state=42, n_jobs=-1, eval_metric="rmse",
    early_stopping_rounds=40, verbosity=0)
# eval_set[0] = fit set (for the "train" curve), eval_set[1] = inner val (early
# stop + "validation" curve). Both come from TRAINING broths. X_test is not here.
model_final.fit(Xf_fit, yf_fit,
                eval_set=[(Xf_fit, yf_fit), (Xf_val, yf_val)], verbose=50)

y_pred_final = model_final.predict(X_test)      # test touched ONCE, here
r2_final   = r2_score(y_test, y_pred_final)
mae_final  = mean_absolute_error(y_test, y_pred_final)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))
print(f"\nFinal (interpolation reference) R2 = {r2_final:.4f}  "
      f"MAE = {mae_final:.4f}  RMSE = {rmse_final:.4f}  trees = {model_final.best_iteration}")

print("\nModel Comparison")
print(f"{'Model':<30}{'R2':>9}{'MAE':>9}")
print("-" * 48)
for name, r2v, maev in [
        ("Global mean (baseline)", r2_mean, np.nan),
        ("Ridge", r2_ridge, mae_ridge), ("Lasso", r2_lasso, mae_lasso),
        ("Elastic Net", r2_enet, mae_enet),
        ("XGBoost (initial)", r2_init, mae_init),
        ("XGBoost (tuned)", r2_final, mae_final)]:
    mae_str = "N/A" if np.isnan(maev) else f"{maev:.4f}"
    print(f"{name:<30}{r2v:>9.4f}{mae_str:>9}")

[0]	validation_0-rmse:0.30584	validation_1-rmse:0.36376
[50]	validation_0-rmse:0.11243	validation_1-rmse:0.23887
[100]	validation_0-rmse:0.08975	validation_1-rmse:0.23279
[150]	validation_0-rmse:0.07667	validation_1-rmse:0.23026
[200]	validation_0-rmse:0.06744	validation_1-rmse:0.22899
[250]	validation_0-rmse:0.06029	validation_1-rmse:0.22760
[300]	validation_0-rmse:0.05441	validation_1-rmse:0.22647
[350]	validation_0-rmse:0.04996	validation_1-rmse:0.22612
[379]	validation_0-rmse:0.04766	validation_1-rmse:0.22604

Final (interpolation reference) R2 = 0.8910  MAE = 0.0618  RMSE = 0.1098  trees = 339

Model Comparison
Model                                R2      MAE
------------------------------------------------
Global mean (baseline)          -0.0000      N/A
Ridge                            0.6425   0.1537
Lasso                            0.6426   0.1536
Elastic Net                      0.6426   0.1536
XGBoost (initial)                0.8728   0.0700
XGBoost (tuned)                  

In [17]:
from scipy.stats import pearsonr, spearmanr

groups = df["broth_id"]

gkf = GroupKFold(n_splits=5)

fold_r2, fold_mae, fold_rmse = [], [], []
fold_pearson, fold_spearman = [], []

print("\nGroupKFold evaluation (true held-out broths):")

for fold_num, (tr_idx, te_idx) in enumerate(
        gkf.split(X, y, groups=groups), start=1):

    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

    groups_tr = groups.iloc[tr_idx]
    groups_te = groups.iloc[te_idx]

    assert len(set(groups_tr).intersection(set(groups_te))) == 0

    held_broths = sorted(set(df.iloc[te_idx]["broth_id"]))
    held_divs = sorted(df.iloc[te_idx]["diversity_level"].unique())

    print(
        f"\nFold {fold_num}: broths={held_broths}  "
        f"diversity levels={held_divs}  n_test={len(X_te):,}"
    )

    # inner validation split from training broths only
    Xg_fit, yg_fit, Xg_val, yg_val = inner_holdout(
        X_tr, y_tr, groups_tr, frac=0.15, seed=42 + fold_num
    )

    m_fold = xgb.XGBRegressor(
        **best_params,
        n_estimators=1000,
        min_child_weight=3,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        eval_metric="rmse",
        early_stopping_rounds=40,
        verbosity=0
    )

    # important: X_te/y_te are NOT in eval_set
    m_fold.fit(
        Xg_fit, yg_fit,
        eval_set=[(Xg_fit, yg_fit), (Xg_val, yg_val)],
        verbose=False
    )

    yp_fold = m_fold.predict(X_te)

    r2_f = r2_score(y_te, yp_fold)
    mae_f = mean_absolute_error(y_te, yp_fold)
    rmse_f = np.sqrt(mean_squared_error(y_te, yp_fold))

    # correlation metrics
    pearson_f = pearsonr(y_te, yp_fold).statistic
    spearman_f = spearmanr(y_te, yp_fold).statistic

    fold_r2.append(r2_f)
    fold_mae.append(mae_f)
    fold_rmse.append(rmse_f)
    fold_pearson.append(pearson_f)
    fold_spearman.append(spearman_f)

    print(
        f"  R²={r2_f:.4f}  MAE={mae_f:.4f}  "
        f"RMSE={rmse_f:.4f}  "
        f"Pearson={pearson_f:.4f}  Spearman={spearman_f:.4f}  "
        f"trees={m_fold.best_iteration}"
    )

print(f"\nGroupKFold summary across {gkf.n_splits} folds:")
print(f"  R²:       mean={np.mean(fold_r2):.4f}  std={np.std(fold_r2):.4f}")
print(f"  MAE:      mean={np.mean(fold_mae):.4f}  std={np.std(fold_mae):.4f}")
print(f"  RMSE:     mean={np.mean(fold_rmse):.4f}  std={np.std(fold_rmse):.4f}")
print(f"  Pearson:  mean={np.mean(fold_pearson):.4f}  std={np.std(fold_pearson):.4f}")
print(f"  Spearman: mean={np.mean(fold_spearman):.4f}  std={np.std(fold_spearman):.4f}")

print(f"\nCompare to stratified-split R²={r2_final:.4f}")
print(
    f"Drop: {r2_final - np.mean(fold_r2):.4f}  "
    "(row-wise interpolation advantage / held-out-broth generalization gap)"
)


GroupKFold evaluation (true held-out broths):

Fold 1: broths=['Broth_005', 'Broth_010', 'Broth_015', 'Broth_020', 'Broth_025', 'Broth_030', 'Broth_035', 'Broth_040', 'Broth_045', 'Broth_050', 'Broth_055', 'Broth_060', 'Broth_065', 'Broth_070', 'Broth_075', 'Broth_080', 'Broth_085', 'Broth_090', 'Broth_095', 'Broth_100', 'Broth_105', 'Broth_110', 'Broth_115', 'Broth_120', 'Broth_125', 'Broth_130', 'Broth_135', 'Broth_140', 'Broth_145', 'Broth_150']  diversity levels=[np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16)]  n_test=15,000
  R²=0.7546  MAE=0.1112  RMSE=0.1629  Pearson=0.8722  Spearman=0.8724  trees=282

Fold 2: broths=['Broth_004', 'Broth_009', 'Broth_014', 'Broth_019', 'Broth_024', 'Broth_029', 'Broth_034', 'Broth_039', 'Broth_044', 'Broth_049', 'Broth_054', 'Broth_059', 'Broth_064', 'Broth_069', 'Broth_074', 'Broth_079', 'Broth_084', 'Broth_089', 'Broth_094', 'Broth_099', 'Broth_104', 'Broth_109', 'Broth_114', 'Broth_119', 'Broth_124', 'Broth_129', 'Broth_134

In [13]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import xgboost as xgb

gss = GroupShuffleSplit(
    n_splits=20,
    test_size=0.20,
    random_state=42
)

r2_scores = []
mae_scores = []
rmse_scores = []

for split_num, (tr_idx, te_idx) in enumerate(gss.split(X, y, groups=groups), start=1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

    groups_tr = groups.iloc[tr_idx] if hasattr(groups, "iloc") else groups[tr_idx]
    groups_te = groups.iloc[te_idx] if hasattr(groups, "iloc") else groups[te_idx]

    # Safety check: no broth overlap between train and test
    assert len(set(groups_tr).intersection(set(groups_te))) == 0

    Xg_fit, yg_fit, Xg_val, yg_val = inner_holdout(
        X_tr, y_tr, groups_tr, frac=0.15, seed=42 + split_num
    )

    m = xgb.XGBRegressor(
        **best_params,
        n_estimators=1000,
        min_child_weight=3,
        reg_lambda=1.0,
        random_state=42 + split_num,
        n_jobs=-1,
        eval_metric="rmse",
        early_stopping_rounds=40,
        verbosity=0
    )

    m.fit(
        Xg_fit, yg_fit,
        eval_set=[(Xg_fit, yg_fit), (Xg_val, yg_val)],
        verbose=False
    )

    yp = m.predict(X_te)

    r2 = r2_score(y_te, yp)
    mae = mean_absolute_error(y_te, yp)
    rmse = np.sqrt(mean_squared_error(y_te, yp))

    r2_scores.append(r2)
    mae_scores.append(mae)
    rmse_scores.append(rmse)

    print(
        f"Split {split_num:02d}: "
        f"R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  "
        f"trees={m.best_iteration}"
    )

print("\nGroupShuffleSplit held-out-broth summary:")
print(f"  R²:   mean={np.mean(r2_scores):.4f}  std={np.std(r2_scores):.4f}")
print(f"  MAE:  mean={np.mean(mae_scores):.4f}  std={np.std(mae_scores):.4f}")
print(f"  RMSE: mean={np.mean(rmse_scores):.4f}  std={np.std(rmse_scores):.4f}")

Split 01: R²=0.1216  MAE=0.2475  RMSE=0.3279  trees=26
Split 02: R²=0.7635  MAE=0.1198  RMSE=0.1644  trees=289
Split 03: R²=0.4770  MAE=0.1703  RMSE=0.2305  trees=117
Split 04: R²=0.6372  MAE=0.1308  RMSE=0.1924  trees=555
Split 05: R²=0.5886  MAE=0.1414  RMSE=0.1986  trees=527
Split 06: R²=0.7118  MAE=0.1249  RMSE=0.1711  trees=365
Split 07: R²=0.7073  MAE=0.1361  RMSE=0.1772  trees=399
Split 08: R²=0.5218  MAE=0.1635  RMSE=0.2258  trees=46
Split 09: R²=0.5504  MAE=0.1761  RMSE=0.2161  trees=19
Split 10: R²=0.5191  MAE=0.1607  RMSE=0.2292  trees=267
Split 11: R²=0.6101  MAE=0.1330  RMSE=0.1978  trees=207
Split 12: R²=0.6157  MAE=0.1486  RMSE=0.2068  trees=301
Split 13: R²=0.6567  MAE=0.1475  RMSE=0.1942  trees=403
Split 14: R²=0.4616  MAE=0.1574  RMSE=0.2385  trees=536
Split 15: R²=0.5854  MAE=0.1574  RMSE=0.2183  trees=362
Split 16: R²=0.8330  MAE=0.1008  RMSE=0.1421  trees=215
Split 17: R²=0.5684  MAE=0.1546  RMSE=0.2255  trees=384
Split 18: R²=0.7329  MAE=0.1202  RMSE=0.1638  trees

In [14]:
print("Performance by diversity level and community size")

results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          y_pred_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

def condition_metrics(y_true_arr, y_pred_arr):
    y_t      = np.asarray(y_true_arr)
    y_p      = np.asarray(y_pred_arr)
    r2       = r2_score(y_t, y_p)
    mae      = mean_absolute_error(y_t, y_p)
    rmse     = np.sqrt(mean_squared_error(y_t, y_p))
    pr, _    = pearsonr(y_t, y_p)
    sr, _    = spearmanr(y_t, y_p)
    bias     = float(np.mean(y_p - y_t))
    std_true = float(np.std(y_t))
    return {'n': len(y_t), 'r2': r2, 'mae': mae, 'rmse': rmse,
            'pearson_r': pr, 'spearman_r': sr,
            'bias': bias, 'std_true': std_true}

hdr = f"{'':6} {'R²':>7} {'MAE':>7} {'RMSE':>7} {'Pearson':>8} {'Spearman':>9} {'Bias':>8} {'std(y)':>7} {'N':>6}"
print("\nBy diversity level:")
print(hdr)
print("-" * len(hdr))
for d, grp in results_df.groupby('diversity_level'):
    m    = condition_metrics(grp['y_true'], grp['y_pred'])
    flag = "  *" if m['std_true'] < 0.08 else "   "
    print(f"D={d:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}{flag}")
print("  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.")

print("\nBy community size:")
print(hdr)
print("-" * len(hdr))
for n, grp in results_df.groupby('community_size'):
    m = condition_metrics(grp['y_true'], grp['y_pred'])
    print(f"N={n:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}")

Performance by diversity level and community size

By diversity level:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
D=1     0.9722  0.0396  0.0722   0.9861    0.9628  -0.0010  0.4330  3,000   
D=2     0.7909  0.0903  0.1630   0.8917    0.8997   0.0196  0.3564  3,000   
D=4     0.7492  0.0994  0.1474   0.8691    0.8607  -0.0172  0.2944  3,000   
D=8     0.9318  0.0486  0.0679   0.9657    0.9639  -0.0005  0.2600  3,000   
D=16    0.9287  0.0314  0.0460   0.9660    0.9710  -0.0046  0.1722  3,000   
  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.

By community size:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
N=12    0.8467  0.0807  0.1282   0.9209    0.9038  -0.0027  0.3274  3,000
N=16    0.8656  0.0753  0.1213   0.9310    0.9270  -0.0029  0.3309  3,000
N=24 

In [15]:
# built in feature importance

print("XGBoost built-in feature importance")

importances = model_final.feature_importances_
importance_series = pd.Series(importances, index=feature_cols).sort_values(ascending=False)

print("\nTop 20 most important features (by gain):")
print(importance_series.head(20).round(5).to_string())

# Summarise importance by feature group
species_imp = importance_series[species_cols].sum()
r0_imp      = importance_series[r0_cols].sum()
active_imp  = importance_series[active_cols].sum()
struct_imp  = importance_series[structural_cols].sum()

print(f"\nImportance grouped by feature type (total):")
print(f"  Species membership:  {species_imp:.4f}  ({100*species_imp:.1f}%)")
print(f"  R0 concentrations:   {r0_imp:.4f}  ({100*r0_imp:.1f}%)")
print(f"  Active flags:        {active_imp:.4f}  ({100*active_imp:.1f}%)")
print(f"  Structural (size/div):{struct_imp:.4f}  ({100*struct_imp:.1f}%)")

XGBoost built-in feature importance

Top 20 most important features (by gain):
active_c8          0.28751
R0_c8              0.06744
active_c9          0.06548
R0_c9              0.04078
R0_c6              0.03640
diversity_level    0.03050
active_c20         0.03030
R0_c20             0.02835
R0_c10             0.02759
R0_c1              0.02413
R0_c7              0.02101
R0_c4              0.02011
R0_c3              0.01997
R0_c16             0.01651
R0_c11             0.01474
active_c6          0.01436
R0_c2              0.01405
R0_c13             0.01198
R0_c19             0.01188
active_c14         0.01130

Importance grouped by feature type (total):
  Species membership:  0.1220  (12.2%)
  R0 concentrations:   0.3966  (39.7%)
  Active flags:        0.4446  (44.5%)
  Structural (size/div):0.0368  (3.7%)


In [16]:
# shapley value analysis

print("SHAP analysis")


explainer = shap.TreeExplainer(model_final)

sample_idx = np.random.default_rng(42).choice(len(X_test), size=2000, replace=False)


X_sample = X_test.iloc[sample_idx]


shap_values = explainer.shap_values(X_sample)
# shap_values: array shape (2000, 142). Each row sums to (prediction - mean).
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)

print("\nTop 20 features by mean |SHAP| value:")
print(shap_importance.head(20).round(5).to_string())

# SHAP by feature group
sp_shap     = shap_importance[species_cols].sum()
r0_shap     = shap_importance[r0_cols].sum()
act_shap    = shap_importance[active_cols].sum()
struct_shap = shap_importance[structural_cols].sum()
total_shap  = sp_shap + r0_shap + act_shap + struct_shap

print(f"\nSHAP importance by feature group (sum of mean |SHAP|):")
print(f"  Species membership:  {sp_shap:.4f}  ({100*sp_shap/total_shap:.1f}%)")
print(f"  R0 concentrations:   {r0_shap:.4f}  ({100*r0_shap/total_shap:.1f}%)")
print(f"  Active flags:        {act_shap:.4f}  ({100*act_shap/total_shap:.1f}%)")
print(f"  Structural:          {struct_shap:.4f}  ({100*struct_shap/total_shap:.1f}%)")

SHAP analysis

Top 20 features by mean |SHAP| value:
R0_c8             0.06699
R0_c6             0.04827
community_size    0.03646
R0_c9             0.02942
Species_0064      0.02781
R0_c16            0.02693
R0_c20            0.02511
R0_c1             0.02128
R0_c17            0.01848
Species_0076      0.01825
R0_c3             0.01771
Species_0080      0.01659
R0_c14            0.01526
R0_c13            0.01501
R0_c10            0.01501
R0_c19            0.01320
R0_c18            0.01319
Species_0084      0.01270
R0_c12            0.01236
Species_0100      0.01221

SHAP importance by feature group (sum of mean |SHAP|):
  Species membership:  0.3084  (40.3%)
  R0 concentrations:   0.3931  (51.4%)
  Active flags:        0.0173  (2.3%)
  Structural:          0.0464  (6.1%)


In [17]:
# figures


print("Generating plots → suppression_results.png")


# plt.figure(figsize=(width, height)): creates a new figure window.
# figsize is in inches. dpi (dots per inch) controls resolution.
fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle('XGBoost Nutrient-Aware Suppression Predictor',
             fontsize=15, fontweight='bold', y=0.98)

# plt.subplot(nrows, ncols, panel_index): creates a subplot in a grid.
# Here we use a 2×3 grid (2 rows, 3 columns), panels numbered 1–6.

# Panel 1: Predicted vs Actual
ax1 = fig.add_subplot(2, 3, 1)
# ax.scatter(x, y): plots individual data points.
# alpha=0.3 makes points semi-transparent so density is visible.
# c sets colour, s sets marker size.
ax1.scatter(y_test, y_pred_final, alpha=0.15, s=3, c='steelblue')
# plot a y=x reference line (perfect predictions would lie on this line)
ax1.plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Perfect prediction')
ax1.set_xlabel('True community suppression')
ax1.set_ylabel('Predicted community suppression')
ax1.set_title(f'Predicted vs Actual  (R²={r2_final:.3f})')
ax1.legend(fontsize=8)
ax1.set_xlim(-0.05, 1.05)
ax1.set_ylim(-0.05, 1.05)

# Panel 2: Residuals
# residuals = y_true - y_pred
# Ideally should be random noise centred at 0.
# Systematic patterns (e.g. always under-predicting at high suppression)
# suggest the model is missing something.
residuals = y_test.values - y_pred_final
ax2 = fig.add_subplot(2, 3, 2)
ax2.scatter(y_pred_final, residuals, alpha=0.15, s=3, c='darkorange')
ax2.axhline(0, color='r', linestyle='--', linewidth=1.5)
# np.percentile(array, [5, 95]): returns the 5th and 95th percentile values.
p5, p95 = np.percentile(residuals, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', linewidth=1, label=f'5th pct: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', linewidth=1, label=f'95th pct: {p95:.3f}')
ax2.set_xlabel('Predicted suppression')
ax2.set_ylabel('Residual (true − predicted)')
ax2.set_title('Residual Plot')
ax2.legend(fontsize=7)

# Panel 3: Top 20 SHAP features
ax3 = fig.add_subplot(2, 3, 3)
top20_shap = shap_importance.head(20)
colours = []
for feat in top20_shap.index:
    if feat in species_cols:  colours.append('#2196F3')   # blue = species
    elif feat in r0_cols:     colours.append('#4CAF50')   # green = R0
    elif feat in active_cols: colours.append('#FF9800')   # orange = active
    else:                     colours.append('#9C27B0')   # purple = structural
# barh: horizontal bar chart. top20_shap.values = bar lengths.
# [::-1] reverses so highest is at top.
ax3.barh(range(20), top20_shap.values[::-1], color=colours[::-1])
ax3.set_yticks(range(20))
ax3.set_yticklabels(top20_shap.index[::-1], fontsize=7)
ax3.set_xlabel('Mean |SHAP value|')
ax3.set_title('Top 20 Features (SHAP)')

# Add a legend for colour coding
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', label='Species'),
    Patch(facecolor='#4CAF50', label='R0'),
    Patch(facecolor='#FF9800', label='Active'),
    Patch(facecolor='#9C27B0', label='Structural'),
]
ax3.legend(handles=legend_elements, loc='lower right', fontsize=7)

# Panel 4: R² by diversity level
ax4 = fig.add_subplot(2, 3, 4)
div_levels = sorted(results_df['diversity_level'].unique())
r2_by_div = [r2_score(
    results_df[results_df['diversity_level']==d]['y_true'],
    results_df[results_df['diversity_level']==d]['y_pred']
) for d in div_levels]
ax4.bar([str(d) for d in div_levels], r2_by_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', linewidth=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level (# active nutrients)')
ax4.set_ylabel('R²')
ax4.set_title('Model R² by Diversity Level')
ax4.set_ylim(0, 1.05)
ax4.legend(fontsize=8)

# Panel 5: R² by community size 
ax5 = fig.add_subplot(2, 3, 5)
sizes = sorted(results_df['community_size'].unique())
r2_by_size = [r2_score(
    results_df[results_df['community_size']==n]['y_true'],
    results_df[results_df['community_size']==n]['y_pred']
) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_by_size, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', linewidth=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N')
ax5.set_ylabel('R²')
ax5.set_title('Model R² by Community Size')
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=8)

# Panel 6: SHAP summary by feature group (pie)
ax6 = fig.add_subplot(2, 3, 6)
group_names   = ['Species\nmembership', 'R0\nconcentrations', 'Active\nflags', 'Structural']
group_shap    = [sp_shap, r0_shap, act_shap, struct_shap]
group_colours = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
# ax.pie: creates a pie chart.
# autopct='%.1f%%': formats each slice's label as a percentage with 1 decimal.
# startangle: rotates the starting position.
wedges, texts, autotexts = ax6.pie(
    group_shap, labels=group_names, colors=group_colours,
    autopct='%.1f%%', startangle=90, textprops={'fontsize': 9}
)
ax6.set_title('SHAP Importance by Feature Group')

plt.tight_layout(rect=[0, 0, 1, 0.97])
# tight_layout: automatically adjusts subplot spacing to avoid overlap.
# rect=[left, bottom, right, top] reserves space for the suptitle.

plt.savefig('suppression_results_xgb01.png', dpi=150, bbox_inches='tight')
# bbox_inches='tight': crops the saved figure to the actual content boundary
# (removes whitespace margins). Always use this.
plt.close()
# plt.close(): frees memory by closing the figure after saving.

print("Saved: suppression_results.png")

# Expanded global metrics

def full_model_metrics(y_true, y_pred, label):
    y_t   = np.asarray(y_true)
    y_p   = np.asarray(y_pred)
    resid = np.abs(y_t - y_p)
    r2    = r2_score(y_t, y_p)
    mae   = mean_absolute_error(y_t, y_p)
    rmse  = np.sqrt(mean_squared_error(y_t, y_p))
    pr, _ = pearsonr(y_t, y_p)
    sr, _ = spearmanr(y_t, y_p)
    return {
        'label':      label,
        'r2':         r2,
        'mae':        mae,
        'rmse':       rmse,
        'pearson_r':  pr,
        'spearman_r': sr,
        'median_ae':  float(np.median(resid)),
        'max_ae':     float(np.max(resid)),
        'within_05':  float(np.mean(resid <= 0.05)),
        'within_10':  float(np.mean(resid <= 0.10)),
    }

metrics_rows = [
    full_model_metrics(y_test, y_pred_mean,  'Global mean (floor)'),
    full_model_metrics(y_test, y_pred_ridge, 'Ridge'),
    full_model_metrics(y_test, y_pred_lasso, 'Lasso'),
    full_model_metrics(y_test, y_pred_enet,  'Elastic Net'),
    full_model_metrics(y_test, y_pred_init,  'XGBoost initial'),
    full_model_metrics(y_test, y_pred_final, 'XGBoost tuned'),
]

print("Expanded global metrics")
hdr = (f"{'Model':<22} {'R2':>7} {'MAE':>7} {'RMSE':>7} "
       f"{'Pearson':>8} {'Spearman':>9} {'MedAE':>7} "
       f"{'MaxAE':>7} {'W.05':>6} {'W.10':>6}")
print(hdr)
print("-" * len(hdr))
for m in metrics_rows:
    print(f"{m['label']:<22} {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['median_ae']:>7.4f} {m['max_ae']:>7.4f} "
          f"{m['within_05']:>6.3f} {m['within_10']:>6.3f}")

Generating plots → suppression_results.png
Saved: suppression_results.png
Expanded global metrics
Model                       R2     MAE    RMSE  Pearson  Spearman   MedAE   MaxAE   W.05   W.10
-----------------------------------------------------------------------------------------------
Global mean (floor)    -0.0000  0.2916  0.3324      nan       nan  0.3017  0.5169  0.080  0.163
Ridge                   0.6425  0.1537  0.1987   0.8016    0.8156  0.1207  0.9235  0.220  0.427
Lasso                   0.6426  0.1536  0.1987   0.8016    0.8156  0.1206  0.9222  0.222  0.429
Elastic Net             0.6426  0.1536  0.1987   0.8016    0.8156  0.1206  0.9222  0.222  0.429
XGBoost initial         0.8728  0.0700  0.1185   0.9349    0.9353  0.0382  0.8909  0.593  0.795
XGBoost tuned           0.8910  0.0618  0.1098   0.9442    0.9444  0.0320  0.9122  0.650  0.832


In [18]:
# save model

model_final.save_model('xgb01_model.json')
print("Model saved: xgb01_model.json")

# Save predictions alongside true values for downstream analysis.
pred_df = pd.DataFrame({
    'community_id':       df.loc[X_test.index, 'community_id'].values,
    'broth_id':           df.loc[X_test.index, 'broth_id'].values,
    'diversity_level':    df.loc[X_test.index, 'diversity_level'].values,
    'community_size':     df.loc[X_test.index, 'community_size'].values,
    'y_true':             y_test.values,
    'y_pred':             y_pred_final,
    'residual':           y_test.values - y_pred_final,
})
pred_df.to_csv('predictions_xgb01.csv', index=False)
print("Predictions saved: predictions_xgb01.csv")

# Save SHAP values for the 2000-sample subset.
shap_df = pd.DataFrame(shap_values, columns=feature_cols)
shap_df.to_csv('shap_values_xgb01.csv', index=False)
print("SHAP values saved: shap_values_xgb01.csv")



Model saved: xgb01_model.json
Predictions saved: predictions_xgb01.csv
SHAP values saved: shap_values_xgb01.csv


In [19]:
# summary

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Task:     predict community_suppression from species membership + nutrient diversity information + community size information")
print(f"Dataset:  30,000 communities × 142 features (v8)")
print(f"")
print(f"Model performance on held-out test set (20%):")
print(f"  R²    = {r2_final:.4f}")
print(f"  MAE   = {mae_final:.4f}  (average error in [0,1] suppression units)")
print(f"  RMSE  = {rmse_final:.4f}")
print(f"")
print(f"Comparison:")
# In the Comparison block at the bottom, change:
print(f"  Global mean baseline     R² = 0.0000  (reference floor)")
print(f"  Ridge Regression         R² = {r2_ridge:.4f}  (linear, no interactions)")
print(f"  Lasso                    R² = {r2_lasso:.4f}  (sparse linear, {n_lasso_kept} features kept)")
print(f"  Elastic Net              R² = {r2_enet:.4f}  (sparse linear with grouping)")
print(f"  XGBoost (tuned)          R² = {r2_final:.4f}  (captures non-linearities)")
print(f"  GroupKFold (held broths) R² = {np.mean(fold_r2):.4f} +/- {np.std(fold_r2):.4f}  (generalisation estimate)")


FINAL SUMMARY
Task:     predict community_suppression from species membership + nutrient diversity information + community size information
Dataset:  30,000 communities × 142 features (v8)

Model performance on held-out test set (20%):
  R²    = 0.8910
  MAE   = 0.0618  (average error in [0,1] suppression units)
  RMSE  = 0.1098

Comparison:
  Global mean baseline     R² = 0.0000  (reference floor)
  Ridge Regression         R² = 0.6425  (linear, no interactions)
  Lasso                    R² = 0.6426  (sparse linear, 134 features kept)
  Elastic Net              R² = 0.6426  (sparse linear with grouping)
  XGBoost (tuned)          R² = 0.8910  (captures non-linearities)
  GroupKFold (held broths) R² = 0.6166 +/- 0.0981  (generalisation estimate)
